# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Output the dataset title and description
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.


In [ ]:
# List all record sets with their @id and fields
if not metadata.record_set:
    print("No record sets were found in the Croissant metadata.")
else:
    print("Record sets found:")
    for record_set in metadata.record_set:
        print(f"- RecordSet @id: {record_set['@id']}, name: {record_set.get('name', 'N/A')}")
        if 'field' in record_set:
            fields = record_set['field']
            if not isinstance(fields, list):
                fields = [fields]
            for field in fields:
                if isinstance(field, dict):
                    print(f"    - Field @id: {field.get('@id', str(field))}, name: {field.get('name', 'N/A')}")
                else:
                    print(f"    - Field @id: {field}")
        else:
            print("    No fields found in this record set.")

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.


In [ ]:
# Extract data from each record set
record_set_ids = []
if metadata.record_set:
    for record_set in metadata.record_set:
        rsid = record_set["@id"]
        record_set_ids.append(rsid)
else:
    print("No record sets available.")

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
    except Exception as ex:
        print(f"Skipping record set {record_set_id} due to error: {ex}")
        continue
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set: {record_set_id} (columns={df.columns.tolist()})")
    else:
        print(f"Record set {record_set_id}: no records found.")

# Preview data for the main record set (if present)
main_rs_id = None
if dataframes:
    main_rs_id = next(iter(dataframes.keys()))
    print(f"\nColumns in main record set ({main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No dataframes created (check prior step for success).")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.


In [ ]:
import numpy as np

# Identify a numeric field to analyze
numeric_field = None
if main_rs_id and not dataframes[main_rs_id].empty:
    # Try to find a column that looks numeric
    df = dataframes[main_rs_id]
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_candidates:
        # Try to coerce columns to numeric
        for col in df.columns:
            coerced = pd.to_numeric(df[col], errors='coerce')
            if coerced.notnull().sum() > 0:
                # Overwrite with numeric data
                dataframes[main_rs_id][col] = coerced
                numeric_candidates.append(col)
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")

if numeric_field:
    # Filter based on numeric_field (e.g., keep rows > 10 in that field)
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean())/filtered_df[numeric_field].std()
    print(f"\nNormalized values of {numeric_field}:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by a non-numeric field if available
    group_field = None
    for col in df.columns:
        if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col]):
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. This will use the main extracted DataFrame.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered > {threshold})")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
In this notebook, we:
- Loaded and explored the FAIR² Croissant dataset using `mlcroissant` by referencing all entities by their `@id`s.
- Reviewed available record sets and fields.
- Loaded records into pandas DataFrames for flexible data analysis.
- Applied basic filtering and normalization steps on a key numeric field, and explored data grouped by a categorical variable.
- Visualized the distribution of data and relationships between attributes.

**Next steps**: Refine the analysis with more domain-specific techniques and leverage additional record sets or fields as needed.